In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7"

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm

import pandas as pd

model_name = "mistralai/Mistral-7B-v0.1"
# model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True, 
#                                              torch_dtype=torch.float16,
#                                              trust_remote_code=True).cuda()
# tokenizer = AutoTokenizer.from_pretrained(model_name)

### load the test data and setup logging

In [ ]:
df = pd.read_csv('data/real_toxicity_prompts_challenging_set.csv')

In [ ]:
from functools import partial
import sys
sys.path.append('../')
from utils.inference import vanila_inference, StopOnTokens

max_new_tokens = 50
inference_fun = partial(vanila_inference, fschat='', max_new_tokens=max_new_tokens)

In [ ]:
from tqdm import tqdm

In [ ]:
responses = []

for row in tqdm(df.iterrows()):
    prompt = row[1]['prompts']
    continuation = inference_fun(raw_query=prompt, model=model, tokenizer=tokenizer)
    continuation = continuation.split(prompt)[-1].strip().rstrip()
    responses.append({
        'prompt': prompt,
        'continuation': continuation,
        })

In [ ]:
print(len(responses))

### Save the responses just in case

In [ ]:
import json
import os

In [ ]:
# dest_file = os.path.join('results', 'mistral_base_model.jsonl') 
# output_file = open(dest_file, 'w', encoding='utf-8')
# for dic in responses:
#     json.dump(dic, output_file) 
#     output_file.write("\n")

In [ ]:
output_file.close()

### Load the responses

In [ ]:
results_file = os.path.join('results', 'mistral_base_model.jsonl') 
with open(results_file) as json_file:
    responses =  [json.loads(line) for line in json_file]

In [ ]:
len(responses)

In [ ]:
responses[0]

### Now lets evaluate the toxicity scores

In [ ]:
import sys

sys.path.append('./')

from utils import ToxicityMetrics

In [ ]:
metric_calculator = ToxicityMetrics()

In [ ]:
responses_text = [t['continuation'] for t in responses]

In [ ]:
average_toxicity, p_toxicity = metric_calculator.calculate_toxicity(responses_text)

In [ ]:
print(f'average toxicity score = {average_toxicity}')

In [ ]:
print(f'percentage toxicity = {p_toxicity*100}%')

### Now lets evaluate the perplexity

In [ ]:
pred_sentences = [item['prompt'].strip().rstrip() + " " + item['continuation'].strip().rstrip() for item in responses]

In [ ]:
perplexity = metric_calculator.calculate_perplexity(pred_sentences, model_name=model_name)

## Lets evaluate the ngram repetition

In [ ]:
diversity_score = metric_calculator.calculate_diversity(responses_text)

In [ ]:
print(f'diversity score = {diversity_score:.3f}')